# Price Prediction Notebook

This notebook demonstrates loading historical price data, training a model, and forecasting future prices. It also supports Excel and macro feature input.

In [1]:
import json
from datetime import date, timedelta
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Load data

Replace `sample_data.csv` with your CSV or Excel file.

In [2]:
def load_data(file_path):
    ext = Path(file_path).suffix.lower()
    if ext in ['.xlsx', '.xls']:
        df = pd.read_excel(file_path, parse_dates=['date'])
    else:
        df = pd.read_csv(file_path, parse_dates=['date'])
    df = df.sort_values('date').reset_index(drop=True)
    return df

sample_path = 'sample_data.csv'
df = load_data(sample_path)
df.head()

,date,price,gdp,interest_rate,unemployment_rate,sales_volume
0,2020-01-01,100,2.1,1.5,3.8,12000
1,2020-02-01,102,2.0,1.5,3.9,11800
2,2020-03-01,98,1.8,1.4,4.2,11200
3,2020-04-01,95,1.6,1.3,4.8,10500
4,2020-05-01,97,1.7,1.3,4.5,10800


## Fetch from Yahoo Finance or manual entry

In [ ]:
def fetch_yahoo_data(ticker, start_date, end_date):
    df = yf.download(ticker, start=start_date, end=end_date, progress=False)
    df = df.reset_index()
    df = df.rename(columns={'Date': 'date', 'Close': 'price'})
    df = df.sort_values('date').reset_index(drop=True)
    return df


def parse_manual_entry(manual_text):
    df = pd.read_csv(StringIO(manual_text), parse_dates=['date'])
    df = df.sort_values('date').reset_index(drop=True)
    return df

# Example Yahoo Finance fetch
# df = fetch_yahoo_data('AAPL', '2020-01-01', '2024-01-01')
# df.head()

# Example manual entry
# manual_text = 'date,price\n2024-01-01,100\n2024-02-01,105'
# df = parse_manual_entry(manual_text)
# df.head()

## Feature engineering

Create time features, lags, rolling averages and price differences.

In [ ]:
def add_time_features(df):
    df = df.copy()
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['quarter'] = df['date'].dt.quarter
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    return df

def feature_engineering(df, drop_na=True):
    df = add_time_features(df)
    df['price_lag_1'] = df['price'].shift(1)
    df['price_lag_2'] = df['price'].shift(2)
    df['price_lag_3'] = df['price'].shift(3)
    df['price_roll_3'] = df['price'].rolling(window=3, min_periods=1).mean()
    df['price_roll_6'] = df['price'].rolling(window=6, min_periods=1).mean()
    df['price_diff_1'] = df['price'].diff(1)
    if drop_na:
        df = df.dropna()
    return df.reset_index(drop=True)

df_fe = feature_engineering(df)
df_fe.head()

## Train the model

Use a pipeline with imputation, scaling, and a gradient boosting regressor.

In [4]:
def build_model():
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, max_depth=6, early_stopping=True, random_state=42)),
    ])

X = df_fe.drop(columns=['date', 'price'])
y = df_fe['price']
model = build_model()
model.fit(X, y)

predictions = model.predict(X)
mae = np.mean(np.abs(predictions - y))
rmse = np.sqrt(np.mean((predictions - y) ** 2))
mae, rmse

(np.float64(14.64197495884283), np.float64(17.00545141123811))

## Forecast future prices

Set the forecast horizon and optional manual macro inputs.

In [5]:
def make_future_dates(last_date, periods, frequency):
    if frequency.lower() in ['d', 'day', 'daily']:
        return pd.date_range(start=last_date + timedelta(days=1), periods=periods, freq='D')
    if frequency.lower() in ['w', 'week', 'weekly']:
        return pd.date_range(start=last_date + timedelta(days=7), periods=periods, freq='W')
    if frequency.lower() in ['m', 'month', 'monthly']:
        return pd.date_range(start=last_date + pd.offsets.MonthBegin(1), periods=periods, freq='MS')
    if frequency.lower() in ['q', 'quarter', 'quarterly']:
        return pd.date_range(start=last_date + pd.offsets.QuarterBegin(startingMonth=1), periods=periods, freq='QS')
    return pd.date_range(start=last_date + timedelta(days=1), periods=periods, freq='D')

future_dates = make_future_dates(df['date'].iloc[-1], 6, 'm')
future_dates

DatetimeIndex(['2024-01-01', '2024-02-01', '2024-03-01', '2024-04-01',
               '2024-05-01', '2024-06-01'],
              dtype='datetime64[us]', freq='MS')

In [6]:
def forecast_macro_by_growth(df, future_dates, macro_cols):
    history = df.set_index('date')[macro_cols].dropna(how='all')
    growth = history.pct_change().mean()
    current = history.iloc[-1].copy()
    rows = []
    for _ in range(len(future_dates)):
        current = current * (1 + growth.fillna(0))
        rows.append(current.to_dict())
    return pd.DataFrame(rows, index=future_dates)

macro_cols = [c for c in df.columns if c not in ['date', 'price']]
future_macro = forecast_macro_by_growth(df, future_dates, macro_cols)
future_macro.head()

,gdp,interest_rate,unemployment_rate,sales_volume
2024-01-01,4.576503,5.341341,1.471586,18622.074660
2024-02-01,4.654307,5.486524,1.443711,18795.754181
2024-03-01,4.733434,5.635654,1.416364,18971.053531
2024-04-01,4.813906,5.788836,1.389534,19147.987818
2024-05-01,4.895746,5.946183,1.363213,19326.572289


In [ ]:
def predict_future(model, df, future_horizon, frequency, future_macro=None):
    last_row = df.iloc[-1] 
    future_dates = make_future_dates(last_row['date'], future_horizon, frequency)
    rows = []
    current = last_row.copy()
    for dt in future_dates:
        row = current.copy()
        row['date'] = dt
        row['price'] = np.nan
        if future_macro is not None and dt in future_macro.index:
            for col in future_macro.columns:
                row[col] = future_macro.loc[dt, col] 
        rows.append(row)
        current = row
    future_df = pd.DataFrame(rows).reset_index(drop=True)
    full_df = pd.concat([df, future_df], ignore_index=True, sort=False)
    full_df = feature_engineering(full_df, drop_na=False)
    future_slice = full_df[full_df['date'].isin(future_dates)].copy()
    future_X = future_slice.drop(columns=['date', 'price'])
    future_slice['predicted_price'] = model.predict(future_X)
    return future_slice[['date', 'predicted_price'] + [c for c in future_slice.columns if c not in ['date', 'price', 'predicted_price']]]

predictions = predict_future(model, df, 6, 'm', future_macro)
predictions

## Save predictions

Export your forecast to CSV.

In [ ]:
predictions.to_csv('notebook_predictions.csv', index=False)